In [1]:
import os
import random
import time
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from PIL import Image, ImageChops

from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
import timm

from scipy.optimize import minimize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
REG_MAX = 4.5
threshold = [
    0.75 * (REG_MAX / 4.0),
    1.5 * (REG_MAX / 4.0),
    2.5 * (REG_MAX / 4.0),
    3.5 * (REG_MAX / 4.0),
]


def regress2class(out):
    prediction = torch.zeros(out.size(0), device=out.device)
    for i in range(4):
        prediction += (out.data >= threshold[i]).squeeze()
    return prediction.detach().cpu()


def regress2class_with_thresholds(out_1d: torch.Tensor, thr):
    thr_t = torch.as_tensor(thr, device=out_1d.device, dtype=out_1d.dtype).view(1, -1)
    pred = (out_1d.view(-1, 1) >= thr_t).sum(dim=1)
    return pred.detach().cpu().numpy().astype(np.int64)


def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5, device=out.device)
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze()
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    pred_prob = torch.zeros((out.size(0), 5), device=out.device)
    for i in range(out.size(0)):
        if out[i] < 4.0:
            l1 = int(math.floor(float(out[i])))
            l2 = int(math.ceil(float(out[i])))
            pred_prob[i][l1] = 1 - (out[i] - l1)
            pred_prob[i][l2] = 1 - (l2 - out[i])
        else:
            pred_prob[i][4] = 1.0
    return pred_prob




In [3]:
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return (
            self.__class__.__name__
            + "("
            + "p="
            + "{:.4f}".format(self.p.data.tolist()[0])
            + ", "
            + "eps="
            + str(self.eps)
            + ")"
        )


# Change (score improvement, minimal & architecture-preserving):
# If no external checkpoint is available, using ImageNet-pretrained EfficientNet weights
# dramatically improves feature quality for this task without changing layers/heads/loss/loops.
# When a checkpoint IS available, we keep pretrained=False to preserve checkpoint semantics.
class Regressor(nn.Module):
    def __init__(self, pretrained_backbone: bool = False):
        super(Regressor, self).__init__()
        self.backbone = timm.models.tf_efficientnet_b5_ns(
            pretrained=pretrained_backbone
        )
        self.backbone.global_pool = GeM(flatten=True)
        self.regressor = nn.Linear(1000, 1)

    def forward(self, x):
        x = self.backbone(x)
        out = self.regressor(x)
        out = torch.sigmoid(out) * 4.5
        return out


class ThreeStage_Model(nn.Module):
    def __init__(self, backbone=None, pretrained_backbone: bool = False):
        super(ThreeStage_Model, self).__init__()

        self.backbone = timm.models.tf_efficientnet_b4_ns(
            pretrained=pretrained_backbone
        )
        self.backbone.global_pool = GeM(flatten=True)

        self.classifier = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 5),
        )

        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 1),
        )

        self.ordinal = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 4),
        )

        self.final_regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x, final=False):
        x = self.backbone(x)

        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)

        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.5
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.5
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out




In [4]:
class photometric_distort(object):
    def __call__(self, image):
        distortions = [
            FT.adjust_brightness,
            FT.adjust_contrast,
            FT.adjust_saturation,
            FT.adjust_hue,
        ]
        random.shuffle(distortions)

        for d in distortions:
            if random.random() < 0.5:
                if d.__name__ == "adjust_hue":
                    adjust_factor = random.uniform(-16 / 255.0, 16 / 255.0)
                else:
                    adjust_factor = random.uniform(0.7, 1.3)
                image = d(image, adjust_factor)
        return image


class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size
        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w

        left = (w - new_w) / 2
        top = (h - new_h) / 2
        right = left + new_w
        bottom = top + new_h
        return image.crop((left, top, right, bottom))


class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0, 0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
            return image.crop(bbox)
        return image




In [5]:
BASE = "../input/aptos2019-blindness-detection"
TRAIN_CSV = os.path.join(BASE, "train.csv")
TEST_CSV = os.path.join(BASE, "test.csv")
TRAIN_IMG_DIR = os.path.join(BASE, "train_images")
TEST_IMG_DIR = os.path.join(BASE, "test_images")

test_df = pd.read_csv(TEST_CSV)
test_ids = test_df["id_code"].values

input_size = 384

transform = transforms.Compose(
    [
        trim(),
        cropTo4_3(),
        transforms.Resize((input_size * 3 // 4, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
    ]
)


def _find_checkpoint():
    candidates = [
        "../input/weights/B4_3stage_13epoch_320.pkl",
        "../input/weights/B4_3stage_13epoch_320.pth",
        "../input/weights/B4_3stage_13epoch_320.pt",
        os.path.join(BASE, "weights", "B4_3stage_13epoch_320.pkl"),
        os.path.join(BASE, "weights", "B4_3stage_13epoch_320.pth"),
        os.path.join(BASE, "weights", "B4_3stage_13epoch_320.pt"),
        os.path.join(BASE, "B4_3stage_13epoch_320.pkl"),
        os.path.join(BASE, "B4_3stage_13epoch_320.pth"),
        os.path.join(BASE, "B4_3stage_13epoch_320.pt"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p

    exts = (".pth", ".pt", ".pkl")
    for root, dirs, files in os.walk(BASE):
        rel = os.path.relpath(root, BASE)
        if rel.count(os.sep) >= 2:
            dirs[:] = []
        for fn in files:
            if fn.endswith(exts) and ("3stage" in fn.lower() or "b4" in fn.lower()):
                return os.path.join(root, fn)
    return None


ckpt_path = _find_checkpoint()

# Change (score improvement, minimal & semantics-preserving):
# If no checkpoint exists, initialize backbone with pretrained weights; otherwise keep original behavior.
PRETRAIN_BACKBONE_WHEN_NO_CKPT = ckpt_path is None
net = ThreeStage_Model(pretrained_backbone=PRETRAIN_BACKBONE_WHEN_NO_CKPT).to(device)

if ckpt_path is not None:
    state = torch.load(ckpt_path, map_location="cpu")
    if (
        isinstance(state, dict)
        and "state_dict" in state
        and isinstance(state["state_dict"], dict)
    ):
        state = state["state_dict"]
    if isinstance(state, dict):
        new_state = {}
        for k, v in state.items():
            new_state[k.replace("module.", "")] = v
        state = new_state
    net.load_state_dict(state, strict=True)
net.eval()

USE_FINAL_HEAD = ckpt_path is not None




/tmp/ipykernel_55/4058361436.py:55: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

In [6]:
class RetinopathyDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        df = df.reset_index(drop=True)
        self.ids = df["id_code"].to_numpy()
        self.labels = df["diagnosis"].to_numpy(dtype=np.float32)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return self.ids.shape[0]

    def __getitem__(self, i):
        img_path = os.path.join(self.img_dir, f"{self.ids[i]}.png")
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        y = torch.tensor(self.labels[i], dtype=torch.float32)
        return img, y


train_df_full = pd.read_csv(TRAIN_CSV)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, va_idx = next(sss.split(train_df_full["id_code"], train_df_full["diagnosis"]))
train_df = train_df_full.iloc[tr_idx].reset_index(drop=True)
val_df = train_df_full.iloc[va_idx].reset_index(drop=True)


def _loader_kwargs(batch_size, shuffle):
    cpu = os.cpu_count() or 2
    num_workers = min(8, cpu)
    pin = device.type == "cuda"
    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin,
        persistent_workers=(num_workers > 0),
    )
    if num_workers > 0:
        kwargs["prefetch_factor"] = 4
    return kwargs


def train_fallback_if_needed():
    global net
    if ckpt_path is not None:
        return

    ds = RetinopathyDataset(train_df, TRAIN_IMG_DIR, transform)
    loader = DataLoader(ds, **_loader_kwargs(batch_size=8, shuffle=True))

    net.train()
    optimizer = optim.Adam(net.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    start = time.time()
    max_seconds = 420
    for epoch in range(1):
        for xb, yb in loader:
            if time.time() - start > max_seconds:
                break
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True).view(-1, 1)

            _, r_out, _ = net(xb)
            loss = criterion(r_out, yb)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        if time.time() - start > max_seconds:
            break

    net.eval()


train_fallback_if_needed()


def _labels_to_int(y_like):
    y = np.asarray(y_like)
    y = np.rint(y).astype(np.int64, copy=False)
    return np.clip(y, 0, 4)


@torch.no_grad()
def predict_regression(df_in, img_dir, batch_size=16):
    ds = RetinopathyDataset(df_in, img_dir, transform)
    loader = DataLoader(ds, **_loader_kwargs(batch_size=batch_size, shuffle=False))
    preds = np.empty(len(ds), dtype=np.float32)
    ys = np.empty(len(ds), dtype=np.float32)
    k = 0
    for xb, yb in loader:
        b = xb.size(0)
        xb = xb.to(device, non_blocking=True)

        if USE_FINAL_HEAD:
            out = net(xb, final=True)
            pred_b = out.squeeze(1)
        else:
            _, r_out, _ = net(xb)
            pred_b = r_out.squeeze(1)

        preds[k : k + b] = pred_b.detach().cpu().numpy().astype(np.float32, copy=False)
        ys[k : k + b] = yb.detach().cpu().numpy().astype(np.float32, copy=False)
        k += b
    return preds, ys


def qwk_from_thresholds(y_true, y_pred_reg, thr):
    thr = np.asarray(thr, dtype=np.float64)
    y_true_i = _labels_to_int(y_true)

    y_pred_reg = np.asarray(y_pred_reg, dtype=np.float64)
    y_pred_cls = (
        (y_pred_reg.reshape(-1, 1) >= thr.reshape(1, -1))
        .sum(axis=1)
        .astype(np.int64, copy=False)
    )
    return cohen_kappa_score(y_true_i, y_pred_cls, weights="quadratic")


def _project_monotone(thr, eps=1e-3, lo=0.0, hi=REG_MAX):
    thr = np.asarray(thr, dtype=np.float64).copy()
    thr[0] = np.clip(thr[0], lo, hi)
    for i in range(1, 4):
        thr[i] = max(thr[i], thr[i - 1] + eps)
        thr[i] = min(thr[i], hi)
    return thr


def tune_thresholds_refined(
    y_true, y_pred_reg, init_thr, n_passes=3, init_step=0.25, min_step=0.01
):
    thr = _project_monotone(init_thr)
    best = qwk_from_thresholds(y_true, y_pred_reg, thr)

    step = float(init_step)
    for _ in range(n_passes):
        improved = True
        while improved:
            improved = False
            for i in range(4):
                lo = 0.0 if i == 0 else thr[i - 1] + 1e-3
                hi = REG_MAX if i == 3 else thr[i + 1] - 1e-3
                if hi <= lo:
                    continue

                grid = np.linspace(-2.0, 2.0, 17, dtype=np.float64) * step
                candidates = thr[i] + grid
                candidates = np.clip(candidates, lo, hi)
                candidates = np.unique(candidates)

                local_best = best
                local_t = thr[i]
                for t in candidates:
                    thr_try = thr.copy()
                    thr_try[i] = t
                    thr_try = _project_monotone(thr_try)
                    score = qwk_from_thresholds(y_true, y_pred_reg, thr_try)
                    if score > local_best:
                        local_best = score
                        local_t = float(t)

                if local_best > best:
                    thr[i] = local_t
                    best = local_best
                    improved = True

            if not improved:
                break

        step = max(step / 2.0, min_step)

    return thr.tolist(), float(best)


@torch.no_grad()
def _predict_regression_with_model(model, df_in, img_dir, batch_size=16):
    ds = RetinopathyDataset(df_in, img_dir, transform)
    loader = DataLoader(ds, **_loader_kwargs(batch_size=batch_size, shuffle=False))
    preds = np.empty(len(ds), dtype=np.float32)
    ys = np.empty(len(ds), dtype=np.float32)
    k = 0
    for xb, yb in loader:
        b = xb.size(0)
        xb = xb.to(device, non_blocking=True)

        if USE_FINAL_HEAD:
            out = model(xb, final=True)
            pred_b = out.squeeze(1)
        else:
            _, r_out, _ = model(xb)
            pred_b = r_out.squeeze(1)

        preds[k : k + b] = pred_b.detach().cpu().numpy().astype(np.float32, copy=False)
        ys[k : k + b] = yb.detach().cpu().numpy().astype(np.float32, copy=False)
        k += b
    return preds, ys


def tune_thresholds_from_true_oof(
    df_full,
    init_thr,
    n_splits=4,
    seed=42,
    batch_size=16,
):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_y = df_full["diagnosis"].to_numpy(dtype=np.float32)
    oof_pred = np.zeros(len(df_full), dtype=np.float32)

    if ckpt_path is not None:
        df_full_for_ds = df_full[["id_code", "diagnosis"]].reset_index(drop=True)
        full_pred_reg, _ = _predict_regression_with_model(
            net, df_full_for_ds, TRAIN_IMG_DIR, batch_size=batch_size
        )
        full_pred_reg = full_pred_reg.astype(np.float32, copy=False)

        for fold, (_, va_i) in enumerate(
            skf.split(df_full["id_code"], df_full["diagnosis"]), 1
        ):
            oof_pred[va_i] = full_pred_reg[va_i]
            fold_qwk_default = qwk_from_thresholds(
                oof_y[va_i], oof_pred[va_i], init_thr
            )
            print(
                f"Fold {fold}/{n_splits} default-thr QWK (OOF preds):",
                float(fold_qwk_default),
            )
    else:

        def _fresh_net_like_loaded():
            # Same fallback logic as before, but also benefit from pretrained backbone when no ckpt,
            # which directly improves OOF predictions used for threshold tuning.
            m = ThreeStage_Model(pretrained_backbone=True).to(device)
            m.eval()
            return m

        def _train_one_epoch_fallback(model, df_train, max_seconds=105, batch_size=8):
            ds = RetinopathyDataset(df_train, TRAIN_IMG_DIR, transform)
            loader = DataLoader(
                ds, **_loader_kwargs(batch_size=batch_size, shuffle=True)
            )
            model.train()
            optimizer = optim.Adam(model.parameters(), lr=1e-4)
            criterion = nn.MSELoss()

            start = time.time()
            for xb, yb in loader:
                if time.time() - start > max_seconds:
                    break
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True).view(-1, 1)
                _, r_out, _ = model(xb)
                loss = criterion(r_out, yb)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            model.eval()
            return model

        for fold, (tr_i, va_i) in enumerate(
            skf.split(df_full["id_code"], df_full["diagnosis"]), 1
        ):
            df_tr = df_full.iloc[tr_i].reset_index(drop=True)
            df_va = df_full.iloc[va_i].reset_index(drop=True)

            fold_model = _fresh_net_like_loaded()
            fold_model = _train_one_epoch_fallback(
                fold_model, df_tr, max_seconds=105, batch_size=8
            )

            va_pred_reg, _ = _predict_regression_with_model(
                fold_model, df_va, TRAIN_IMG_DIR, batch_size=batch_size
            )
            oof_pred[va_i] = va_pred_reg.astype(np.float32, copy=False)

            fold_qwk_default = qwk_from_thresholds(
                oof_y[va_i], oof_pred[va_i], init_thr
            )
            print(
                f"Fold {fold}/{n_splits} default-thr QWK (OOF preds):",
                float(fold_qwk_default),
            )

            del fold_model
            if device.type == "cuda":
                torch.cuda.empty_cache()

    def _thr_from_params(p):
        p = np.asarray(p, dtype=np.float64).ravel()
        p = np.clip(p, -12.0, 12.0)
        base = (1.0 / (1.0 + np.exp(-p[0]))) * REG_MAX
        gaps = np.exp(p[1:4]) * 0.05
        thr = np.cumsum(np.concatenate([[base], gaps]))
        thr = _project_monotone(thr, eps=1e-3, lo=0.0, hi=REG_MAX)
        return thr

    def _params_from_thr(thr):
        thr = _project_monotone(thr)
        base = float(thr[0])
        gaps = np.diff(thr)
        base01 = np.clip(base / REG_MAX, 1e-6, 1.0 - 1e-6)
        p0 = np.log(base01 / (1.0 - base01))
        gaps = np.clip(gaps, 1e-6, None)
        p_rest = np.log(gaps / 0.05)
        p = np.concatenate([[p0], p_rest])
        if p.shape[0] != 4:
            p = np.pad(p, (0, 4 - p.shape[0]), mode="constant")
        return p

    thr0, q0 = tune_thresholds_refined(
        _labels_to_int(oof_y),
        oof_pred,
        init_thr,
        n_passes=3,
        init_step=0.25,
        min_step=0.01,
    )
    p0 = _params_from_thr(thr0)

    def _objective(p):
        thr = _thr_from_params(p)
        return -qwk_from_thresholds(oof_y, oof_pred, thr)

    res = minimize(
        _objective,
        p0,
        method="Powell",
        options={"maxiter": 70, "xtol": 1e-3, "ftol": 1e-4, "disp": False},
    )

    thr_best = _thr_from_params(res.x).tolist()
    qwk_best = qwk_from_thresholds(oof_y, oof_pred, thr_best)
    print(
        "OOF QWK after Powell threshold opt:",
        float(qwk_best),
        " (init:",
        float(q0),
        ")",
    )
    return thr_best, float(qwk_best)


if ckpt_path is not None:
    df_full_for_ds = train_df_full[["id_code", "diagnosis"]].reset_index(drop=True)
    full_pred_reg_cached, full_y_cached = _predict_regression_with_model(
        net, df_full_for_ds, TRAIN_IMG_DIR, batch_size=16
    )
    val_pred_reg = full_pred_reg_cached[va_idx]
    val_y = full_y_cached[va_idx]
else:
    val_pred_reg, val_y = predict_regression(val_df, TRAIN_IMG_DIR, batch_size=16)

tuned_thr_single, tuned_qwk_single = tune_thresholds_refined(
    _labels_to_int(val_y),
    val_pred_reg,
    threshold,
    n_passes=3,
    init_step=0.25,
    min_step=0.01,
)
base_qwk = qwk_from_thresholds(_labels_to_int(val_y), val_pred_reg, threshold)

tuned_thr, tuned_qwk_oof = tune_thresholds_from_true_oof(
    train_df_full, threshold, n_splits=4, seed=42, batch_size=16
)

print("Validation QWK with default thresholds (single split):", float(base_qwk))
print("Validation QWK with tuned thresholds (single split):", float(tuned_qwk_single))
print("Using thresholds (true OOF-tuned):", tuned_thr)
print("QWK on full true-OOF predictions (tuned):", float(tuned_qwk_oof))
print("USE_FINAL_HEAD (ckpt present => True):", bool(USE_FINAL_HEAD))
print("ckpt_path:", ckpt_path)
print("Backbone pretrained due to missing ckpt:", bool(PRETRAIN_BACKBONE_WHEN_NO_CKPT))




/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipykernel_55/4058361436.py:55: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow

Fold 1/4 default-thr QWK (OOF preds): 0.8786296233578457


/tmp/ipykernel_55/4058361436.py:55: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Fold 2/4 default-thr QWK (OOF preds): 0.8797135984066125


/tmp/ipykernel_55/4058361436.py:55: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Fold 3/4 default-thr QWK (OOF preds): 0.8559213339033775


/tmp/ipykernel_55/4058361436.py:55: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Fold 4/4 default-thr QWK (OOF preds): 0.8253602559615507
OOF QWK after Powell threshold opt: 0.873567323122731  (init: 0.8734157917362106 )
Validation QWK with default thresholds (single split): 0.8530012249897918
Validation QWK with tuned thresholds (single split): 0.8863927147827968
Using thresholds (true OOF-tuned): [0.9997647376046105, 1.63253598218324, 2.366910982203299, 3.163455943004677]
QWK on full true-OOF predictions (tuned): 0.873567323122731
USE_FINAL_HEAD (ckpt present => True): False
ckpt_path: None
Backbone pretrained due to missing ckpt: True


In [7]:
class TestDataset(Dataset):
    def __init__(self, ids, img_dir, transform):
        self.ids = list(ids)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        idx = self.ids[i]
        img_path = os.path.join(self.img_dir, f"{idx}.png")
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        return idx, img


test_ds = TestDataset(test_ids, TEST_IMG_DIR, transform)
test_loader = DataLoader(test_ds, **_loader_kwargs(batch_size=16, shuffle=False))

pred_ids = []
pred_diag = np.empty(len(test_ds), dtype=np.int64)
k = 0

with torch.no_grad():
    for ids, xb in test_loader:
        xb = xb.to(device, non_blocking=True)

        if USE_FINAL_HEAD:
            out = net(xb, final=True)
            pred_reg = out.squeeze(1)
        else:
            _, r_out, _ = net(xb)
            pred_reg = r_out.squeeze(1)

        pred_cls = regress2class_with_thresholds(pred_reg, tuned_thr)
        b = len(ids)
        pred_diag[k : k + b] = pred_cls
        pred_ids.extend(list(ids))
        k += b

submission_rows = list(zip(pred_ids, pred_diag.tolist()))



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [8]:
df = pd.DataFrame(submission_rows, columns=["id_code", "diagnosis"])
df = df.set_index("id_code").loc[test_df["id_code"]].reset_index()

df.to_csv("submission.csv", index=False)

print(df.head())
print("Wrote submission.csv with shape:", df.shape)
print("Device:", device)
print("ckpt_path:", ckpt_path)
print("REG_MAX used for threshold tuning:", REG_MAX)
print("Initial threshold (scaled):", threshold)
print("Final tuned_thr:", tuned_thr)
print("USE_FINAL_HEAD:", bool(USE_FINAL_HEAD))
print("Backbone pretrained due to missing ckpt:", bool(PRETRAIN_BACKBONE_WHEN_NO_CKPT))

        id_code  diagnosis
0  b460ca9fa26f          0
1  6cee2e148520          0
2  ca6842bfcbc9          0
3  6cbc3dad809c          2
4  a9bc2f892cb3          0
Wrote submission.csv with shape: (367, 2)
Device: cuda
ckpt_path: None
REG_MAX used for threshold tuning: 4.5
Initial threshold (scaled): [0.84375, 1.6875, 2.8125, 3.9375]
Final tuned_thr: [0.9997647376046105, 1.63253598218324, 2.366910982203299, 3.163455943004677]
USE_FINAL_HEAD: False
Backbone pretrained due to missing ckpt: True
